### 3MT API for Input Modalities

In [1]:
!pip install requests

In [6]:
!pip install fastapi uvicorn opencv-python ffmpeg-python torch faster-whisper deep-translator pillow

     ---------------------------------------- 0.0/94.9 kB ? eta -:--:--
     ---------------------------------------- 94.9/94.9 kB 2.7 MB/s eta 0:00:00
     ---------------------------------------- 0.0/62.3 kB ? eta -:--:--
     ---------------------------------------- 62.3/62.3 kB 3.5 MB/s eta 0:00:00
     ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
     --------------- ------------------------ 0.4/1.1 MB 8.7 MB/s eta 0:00:01
     -------------------------- ------------- 0.7/1.1 MB 7.7 MB/s eta 0:00:01
     ---------------------------------------  1.1/1.1 MB 7.9 MB/s eta 0:00:01
     ---------------------------------------- 1.1/1.1 MB 7.9 MB/s eta 0:00:00
     ---------------------------------------- 0.0/42.3 kB ? eta -:--:--
     ---------------------------------------- 42.3/42.3 kB 2.0 MB/s eta 0:00:00
     ---------------------------------------- 0.0/72.0 kB ? eta -:--:--
     ---------------------------------------- 72.0/72.0 kB 3.9 MB/s eta 0:00:00
     ----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.24.0 requires tokenizers!=0.11.3,<0.14,>=0.11.1, but you have tokenizers 0.21.0 which is incompatible.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.8.0 which is incompatible.
black 0.0 requires packaging>=22.0, but you have packaging 21.3 which is incompatible.


In [1]:
!uvicorn filename:app --host 0.0.0.0 --port 8000 --reload

In [ ]:
from fastapi import FastAPI, File, UploadFile
import os
import shutil
import cv2
import ffmpeg
import torch
from faster_whisper import WhisperModel
from deep_translator import GoogleTranslator
from pathlib import Path
from PIL import Image

app = FastAPI()

# Load Whisper model
model = WhisperModel("large-v2", device="cuda" if torch.cuda.is_available() else "cpu", compute_type="fp16")
translator = GoogleTranslator(source='auto', target='en')  # Set target language here

BASE_DIR = "processed_files"
os.makedirs(BASE_DIR, exist_ok=True)

# Function to extract audio from video
def extract_audio(video_path, output_audio_path):
    try:
        ffmpeg.input(video_path).output(output_audio_path, format='mp3', acodec='mp3').run(overwrite_output=True)
        return True
    except Exception as e:
        print(f"Error extracting audio: {e}")
        return False

# Function to extract frames from video
def extract_frames(video_path, frames_folder):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_filename = os.path.join(frames_folder, f"frame_{frame_count:04d}.jpg")
        cv2.imwrite(frame_filename, frame)
        frame_count += 1
    cap.release()

# Function to transcribe audio
def transcribe_audio(audio_path):
    segments, _ = model.transcribe(audio_path, beam_size=5)
    transcription = " ".join([segment.text for segment in segments])
    return transcription

# Function to process uploaded files
@app.post("/process/")
async def process_file(file: UploadFile = File(...)):
    filename = file.filename
    file_ext = filename.split(".")[-1]
    file_path = os.path.join(BASE_DIR, filename)
    
    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    
    response = {"filename": filename, "processed": []}
    
    # Video Processing
    if file_ext in ["mp4", "mkv", "avi", "mov"]:
        video_folder = os.path.join(BASE_DIR, Path(filename).stem)
        os.makedirs(video_folder, exist_ok=True)
        
        # Extract audio
        audio_path = os.path.join(video_folder, "extracted_audio.mp3")
        if extract_audio(file_path, audio_path):
            transcription = transcribe_audio(audio_path)
            translated_text = translator.translate(transcription)
            
            text_file_path = os.path.join(video_folder, "transcription.txt")
            with open(text_file_path, "w") as f:
                f.write(f"Original: {transcription}\nTranslated: {translated_text}")
            
            response["processed"].append("Audio extracted and transcribed")
        
        # Extract frames
        frames_folder = os.path.join(video_folder, "frames")
        os.makedirs(frames_folder, exist_ok=True)
        extract_frames(file_path, frames_folder)
        response["processed"].append("Frames extracted")
    
    # Audio Processing
    elif file_ext in ["mp3", "wav", "m4a"]:
        audio_folder = os.path.join(BASE_DIR, Path(filename).stem)
        os.makedirs(audio_folder, exist_ok=True)
        
        transcription = transcribe_audio(file_path)
        translated_text = translator.translate(transcription)
        
        text_file_path = os.path.join(audio_folder, "transcription.txt")
        with open(text_file_path, "w") as f:
            f.write(f"Original: {transcription}\nTranslated: {translated_text}")
        
        response["processed"].append("Audio transcribed")
    
    # Image Processing (Just saving for now)
    elif file_ext in ["jpg", "jpeg", "png"]:
        image_folder = os.path.join(BASE_DIR, "images")
        os.makedirs(image_folder, exist_ok=True)
        shutil.move(file_path, os.path.join(image_folder, filename))
        response["processed"].append("Image saved")
    
    # Text File Processing (Just saving for now)
    elif file_ext in ["txt"]:
        text_folder = os.path.join(BASE_DIR, "texts")
        os.makedirs(text_folder, exist_ok=True)
        shutil.move(file_path, os.path.join(text_folder, filename))
        response["processed"].append("Text file saved")
    
    return response